# 03 — Il confronto (tentativo 09)

Sulle 301 testate valutate, fold per gruppo: il **riferimento** (il TF-IDF per articolo delle Versioni 1–4, con C
scelto in validazione interna, ricalcolato da `01_riferimento_tfidf` sullo stesso test), le tre **famiglie** (testo,
link, pagina), le **combinazioni** e il modello **"alla NewsGuard"** (`02_modelli`).

Metriche per testata: F1 macro a 5 livelli e accuratezza binaria (affidabile = punteggio ≥ 60), con intervalli al 95%
da un **bootstrap per gruppo** (2000 ricampionamenti) e le differenze con il riferimento sugli stessi ricampionamenti.
Poi i segnali che pesano di più nei modelli link e pagina, e quanto bene si stimano i 9 criteri.

In [ ]:
import numpy as np
import pandas as pd

from comune import CLASSI, CRITERI, RESULTS, metriche_di, per_testata

B = 2000
OUT = RESULTS / "confronto"; OUT.mkdir(parents=True, exist_ok=True)
pr = pd.read_csv(RESULTS / "modelli" / "predizioni_testate.csv").set_index("domain")
if (RESULTS / "modelli_bert" / "predizioni_testate.csv").exists():     # ModernBERT come famiglia del testo (04-05), se c'è
    pb = pd.read_csv(RESULTS / "modelli_bert" / "predizioni_testate.csv").set_index("domain")
    pr = pr.join(pb[[x for x in pb.columns if " | p_l" in x]], how="inner")
pr = pr[pr.valutazione]
modelli = sorted({c.split(" | ")[0] for c in pr.columns if " | p_l" in c})
P = {m: pr[[f"{m} | p_l{k}" for k in range(1, 6)]].to_numpy().argmax(1) for m in modelli}
rif = per_testata(pd.read_csv(RESULTS / "feature_tuned_fold_gruppo_articoli.csv").rename(columns={"y_pred": "y_pred"}))
P = {"riferimento: TF-IDF per articolo (versioni 1-4)": rif.y_pred.reindex(pr.index).to_numpy(), **P}
y = pr.y_true.to_numpy(); g = pr.index.map(pd.read_csv("../../../../../Dataset/tentativi/09_CambioParadigma/versioni/fold_finale.csv").set_index("domain").gruppo).to_numpy()
print(f"{len(pr)} testate valutate, {len(np.unique(g))} gruppi; modelli: {list(P)}")

In [ ]:
gruppi = np.unique(g); pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
camp = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(p, idx):
    m = metriche_di(y[idx], p[idx]); return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(p, i) for i in camp]) for k, p in P.items()}
R = list(P)[0]
righe = {}
for k, p in P.items():
    m = metriche_di(y, p); b = boot[k]
    r = {"F1 macro": m["f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": m["bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5),
         "accuratezza": m["accuratezza"], "tolerant 1": m["tolerant_1"]}
    if k != R:
        d = b - boot[R]
        r.update({"ΔF1 vs riferimento": m["f1_macro"] - metriche_di(y, P[R])["f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5)})
    righe[k] = r
tab = pd.DataFrame(righe).T
tab.to_csv(OUT / "intervalli.csv")
tab.round(3)

## I segnali che pesano di più (modelli link e pagina, media sui 5 fold)

In [ ]:
co = pd.read_csv(RESULTS / "modelli" / "coefficienti_link_pagina.csv")
top = co.groupby(["famiglia", "livello", "segnale"]).peso.agg(["mean", "count"]).reset_index()
top = top[top["count"] >= 3].sort_values("mean", ascending=False).groupby(["famiglia", "livello"]).head(8)
top.to_csv(OUT / "segnali_piu_forti.csv", index=False)
for (fam, liv), x in top.groupby(["famiglia", "livello"]):
    print(f"{fam} -> {liv}: " + ", ".join(f"{s} ({w:+.2f})" for s, w in zip(x.segnale, x["mean"])))

## Quanto bene si stimano i criteri NewsGuard

In [ ]:
cr = pd.read_csv(RESULTS / "modelli" / "criteri.csv", index_col=0)
cr.round(3)